## Libraries and Spark session

In [1]:
import os
from datetime import date

import findspark
findspark.init()

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, when, lit, coalesce, rand, count,
    round as spark_round,
    sum as spark_sum,
    month, year,
    desc, trim, upper, regexp_replace,
    to_timestamp, current_timestamp
)

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

In [2]:
spark = (
    SparkSession.builder \
    .appName("DAT535-Project-NYC311") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 13:40:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/30 13:40:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/30 13:40:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 3.5.0


## Project paths

In [3]:
PROJECT_DIR = "/home/ubuntu/spark-project-data"

BRONZE_DIR = f"{PROJECT_DIR}/bronze"
SILVER_DIR = f"{PROJECT_DIR}/dev/silver"

BRONZE_NYC311_PATH = f"{BRONZE_DIR}/nyc311/nyc311_2020_present.csv"
SILVER_NYC311_PATH = f"{SILVER_DIR}/nyc311/nyc311_silver.parquet"

print("Bronze file exists:", os.path.isfile(BRONZE_NYC311_PATH))

Bronze file exists: True


## Load the Bronze DataSet file in a DataFrame

In [4]:
bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .csv(BRONZE_NYC311_PATH)
)

print(f"Number of sources columns : {len(bronze_df.columns)}")
print("Columns :", bronze_df.columns)
bronze_df.printSchema()

Number of sources columns : 44
Columns : ['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']
root
 |-- Unique Key: string (nullable = true)
 |-- Created Date: string (nullable = true)
 |-- Closed Date:

## Select usefull columns and rename them

In [5]:
# Mapping: original CSV columns -> Lifecycle columns
LIFECYCLE_SOURCE_TO_TARGET = {
    "Created Date": "created_at_raw",
    "Closed Date": "closed_at_raw",
    "Status": "status_raw",
}


def add_lifecycle_input_columns(df):
    """
    Add Lifecycle input columns without removing original columns.
    Existing columns from Fields and other sources are preserved.
    """

    missing_columns = [
        source
        for source, target in LIFECYCLE_SOURCE_TO_TARGET.items()
        if source not in df.columns and target not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Lifecycle source columns not found: {missing_columns}"
        )

    for source, target in LIFECYCLE_SOURCE_TO_TARGET.items():

        # Do not overwrite an existing Lifecycle input column
        if target not in df.columns:

            source_value = trim(
                col(f"`{source}`").cast("string")
            )

            df = df.withColumn(
                target,
                when(
                    source_value.isNull() | (source_value == ""),
                    lit(None).cast("string")
                ).otherwise(source_value)
            )

    return df

## Parse the original timestamp strings

The parser tries the standard NYC 311 format first, then the alternative formats observed during exploration. It only parses values; lifecycle validation and business rules are applied in `apply_lifecycle()` below.

In [6]:
def parse_nyc311_timestamp(column_name: str):
    """Parse common NYC 311 timestamp formats into a Spark timestamp."""
    value = trim(col(column_name))

    # Main format used by NYC 311
    primary = to_timestamp(value, "MM/dd/yyyy hh:mm:ss a")

    # Backup formats in case the format changes in the future
    fallback = coalesce(
        to_timestamp(value, "M/d/yyyy h:mm:ss a"),
        to_timestamp(value, "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(value, "yyyy-MM-dd'T'HH:mm:ss"),
    )

    return coalesce(primary, fallback)

## Normalize Status

In [7]:
def normalize_lifecycle_status(df):
    """Normalize status values while preserving status_raw."""
    status_key = upper(
        trim(regexp_replace(col("status_raw"), r"\s+", " "))
    )

    return df.withColumn(
        "status",
        when(
            status_key.isNull() | (status_key == ""),
            lit(None).cast("string"),
        )
        .when(status_key == "CLOSED", lit("Closed"))
        .when(status_key == "OPEN", lit("Open"))
        .when(status_key == "IN PROGRESS", lit("In Progress"))
        .when(status_key == "PENDING", lit("Pending"))
        .when(status_key == "ASSIGNED", lit("Assigned"))
        .when(status_key == "UNSPECIFIED", lit("Unspecified"))
        .when(status_key == "CANCEL", lit("Cancel"))
        .when(status_key == "EMAIL SENT", lit("Email Sent"))
        .when(status_key == "STARTED", lit("Started"))
        .when(status_key == "COMPLETED", lit("Completed"))
        .otherwise(
            trim(regexp_replace(col("status_raw"), r"\s+", " "))
        ),
    )

## Parsing and validation of Dates

In [8]:
def add_lifecycle_dates(df):
    """Parse timestamps and add date-quality flags."""

    created_parsed = parse_nyc311_timestamp("created_at_raw")
    closed_parsed = parse_nyc311_timestamp("closed_at_raw")

    raw_created_blank = (
        col("created_at_raw").isNull()
        | (trim(col("created_at_raw")) == "")
    )
    raw_closed_blank = (
        col("closed_at_raw").isNull()
        | (trim(col("closed_at_raw")) == "")
    )

    before_1900 = lit("1900-01-01 00:00:00").cast("timestamp")

    df = (
        df
        .withColumn("_created_at_parsed", created_parsed)
        .withColumn("_closed_at_parsed", closed_parsed)

        .withColumn("created_at_missing", raw_created_blank)
        .withColumn("closed_at_missing", raw_closed_blank)

        .withColumn(
            "created_at_parse_failed",
            (~raw_created_blank) & col("_created_at_parsed").isNull(),
        )
        .withColumn(
            "closed_at_parse_failed",
            (~raw_closed_blank) & col("_closed_at_parsed").isNull(),
        )

        .withColumn(
            "created_at_before_1900",
            coalesce(
                col("_created_at_parsed").isNotNull()
                & (col("_created_at_parsed") < before_1900),
                lit(False),
            ),
        )
        .withColumn(
            "closed_at_before_1900",
            coalesce(
                col("_closed_at_parsed").isNotNull()
                & (col("_closed_at_parsed") < before_1900),
                lit(False),
            ),
        )

        .withColumn(
            "created_at_is_future",
            coalesce(
                col("_created_at_parsed") > current_timestamp(),
                lit(False),
            ),
        )
        .withColumn(
            "closed_at_is_future",
            coalesce(
                col("_closed_at_parsed") > current_timestamp(),
                lit(False),
            ),
        )

        .withColumn(
            "closed_before_created",
            coalesce(
                col("_closed_at_parsed").isNotNull()
                & col("_created_at_parsed").isNotNull()
                & (
                    col("_closed_at_parsed")
                    < col("_created_at_parsed")
                )
                & (~col("closed_at_before_1900")),
                lit(False),
            ),
        )

        .withColumn(
            "created_at",
            when(
                col("created_at_before_1900"),
                lit(None).cast("timestamp"),
            ).otherwise(col("_created_at_parsed")),
        )
        .withColumn(
            "closed_at",
            when(
                col("closed_at_before_1900"),
                lit(None).cast("timestamp"),
            ).otherwise(col("_closed_at_parsed")),
        )
    )

    return df

## Detect `Status` and Dates inconsistency

In [9]:
def check_status_date_consistency(df):
    """Flag inconsistencies between status and closing date."""

    df = (
        df
        .withColumn(
            "closed_status_missing_closed_date",
            coalesce(
                (col("status") == "Closed")
                & col("closed_at_missing"),
                lit(False),
            ),
        )
        .withColumn(
            "non_closed_status_has_closed_date",
            coalesce(
                col("status").isNotNull()
                & (col("status") != "Closed")
                & col("_closed_at_parsed").isNotNull()
                & (~col("created_at_before_1900"))
                & (~col("closed_at_before_1900")),
                lit(False),
            ),
        )
        .withColumn(
            "status_date_consistency",
            when(
                col("status").isNull(),
                lit("missing_status"),
            )
            .when(
                col("closed_at_before_1900"),
                lit("closed_at_before_1900"),
            )
            .when(
                col("closed_at_parse_failed"),
                lit("invalid_closed_date"),
            )
            .when(
                col("closed_at_is_future"),
                lit("future_closed_at"),
            )
            .when(
                col("closed_status_missing_closed_date"),
                lit("closed_status_missing_closed_date"),
            )
            .when(
                col("non_closed_status_has_closed_date"),
                lit("non_closed_status_has_closed_date"),
            )
            .otherwise(lit("consistent")),
        )
    )

    return df

## Add derivated columns from Dates

In [10]:
def add_request_date_parts(df):
    """Derive year and month from the cleaned creation timestamp."""

    return (
        df
        .withColumn(
            "request_year",
            year(col("created_at")).cast("int"),
        )
        .withColumn(
            "request_month",
            month(col("created_at")).cast("int"),
        )
    )

## Add `resolution_hours`

In [11]:
def calculate_resolution_hours(df):
    """Calculate duration only for valid, closed requests."""

    valid_resolution = (
        (col("status") == "Closed")
        & col("created_at").isNotNull()
        & col("closed_at").isNotNull()
        & (~col("created_at_is_future"))
        & (~col("closed_at_is_future"))
        & (~col("closed_before_created"))
        & (col("closed_at") > col("created_at"))
    )

    return df.withColumn(
        "resolution_hours",
        when(
            valid_resolution,
            spark_round(
                (
                    col("closed_at").cast("long")
                    - col("created_at").cast("long")
                ) / lit(3600.0),
                2,
            ).cast("double"),
        ).otherwise(lit(None).cast("double")),
    )

## Add `lifecycle_quality_status`

In [12]:
def add_lifecycle_quality_status(df):
    """Assign one primary quality label to each request."""

    return df.withColumn(
        "lifecycle_quality_status",
        when(
            col("created_at_missing"),
            lit("missing_created_at"),
        )
        .when(
            col("created_at_parse_failed"),
            lit("invalid_created_at"),
        )
        .when(
            col("created_at_before_1900"),
            lit("created_at_before_1900"),
        )
        .when(
            col("created_at_is_future"),
            lit("future_created_at"),
        )
        .when(
            col("closed_at_before_1900"),
            lit("closed_at_before_1900"),
        )
        .when(
            col("closed_at_parse_failed"),
            lit("invalid_closed_at"),
        )
        .when(
            col("closed_at_is_future"),
            lit("future_closed_at"),
        )
        .when(
            col("closed_before_created"),
            lit("closed_before_created"),
        )
        .when(
            col("status").isNull(),
            lit("missing_status"),
        )
        .when(
            col("closed_status_missing_closed_date"),
            lit("closed_status_missing_closed_date"),
        )
        .when(
            col("non_closed_status_has_closed_date"),
            lit("non_closed_status_has_closed_date"),
        )
        .otherwise(lit("ok")),
    )

## Main function

In [13]:
def apply_lifecycle(input_df):
    """
    Apply all Lifecycle transformations in sequence.

    Keeps all original input columns and adds lifecycle columns.
    """
    
    # Add Lifecycle source columns if they do not already exist
    df = add_lifecycle_input_columns(input_df)
    
    required_columns = [
        "created_at_raw",
        "closed_at_raw",
        "status_raw",
    ]

    missing_columns = [
        name
        for name in required_columns
        if name not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Lifecycle input is missing required columns: {missing_columns}"
        )

    df = normalize_lifecycle_status(df)
    df = add_lifecycle_dates(df)
    df = check_status_date_consistency(df)
    df = add_request_date_parts(df)
    df = calculate_resolution_hours(df)
    df = add_lifecycle_quality_status(df)

    # Remove only temporary columns created by Lifecycle
    df = df.drop(
        "_created_at_parsed",
        "_closed_at_parsed"
    )

    return df

In [14]:
nyc311_lifecycle_df = apply_lifecycle(bronze_df)

print("Columns :", nyc311_lifecycle_df.columns)

Columns : ['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location', 'created_at_raw', 'closed_at_raw', 'status_raw', 'created_at_missing', 'closed_at_missing', 'created_at_parse_failed', 'closed_at_parse_fai

## Data quality summary

In [15]:
quality_summary_df = nyc311_lifecycle_df.agg(

    count("*").alias("total_requests"),

    # Missing dates
    spark_sum(when(col("created_at_missing") == True, 1).otherwise(0))
     .alias("missing_created_at"),

    spark_sum(when(col("closed_at_missing") == True, 1).otherwise(0))
     .alias("missing_closed_at"),

    # Failed timestamp parsing
    spark_sum(when(col("created_at_parse_failed") == True, 1).otherwise(0))
     .alias("created_at_parse_failed"),

    spark_sum(when(col("closed_at_parse_failed") == True, 1).otherwise(0))
     .alias("closed_at_parse_failed"),

    # Dates before 1900
    spark_sum(when(col("created_at_before_1900") == True, 1).otherwise(0))
     .alias("created_at_before_1900"),

    spark_sum(when(col("closed_at_before_1900") == True, 1).otherwise(0))
     .alias("closed_at_before_1900"),

    # Future dates
    spark_sum(when(col("created_at_is_future") == True, 1).otherwise(0))
     .alias("future_created_at"),

    spark_sum(when(col("closed_at_is_future") == True, 1).otherwise(0))
     .alias("future_closed_at"),

    # Date inconsistencies
    spark_sum(when(col("closed_before_created") == True, 1).otherwise(0))
     .alias("closed_before_created"),

    spark_sum(when(col("closed_status_missing_closed_date") == True, 1).otherwise(0))
     .alias("closed_status_missing_date"),

    spark_sum(when(col("non_closed_status_has_closed_date") == True, 1).otherwise(0))
     .alias("non_closed_status_has_closed_date"),

    # Resolution duration
    spark_sum(when(col("resolution_hours").isNotNull(), 1).otherwise(0))
     .alias("requests_with_resolution_hours"),

    spark_sum(when(col("resolution_hours").isNull(), 1).otherwise(0))
     .alias("requests_without_resolution_hours")

)

quality_summary_df.show(truncate=False, vertical=True)

[Stage 1:======================================================>(119 + 1) / 120]

-RECORD 0-------------------------------------
 total_requests                    | 22761196 
 missing_created_at                | 0        
 missing_closed_at                 | 439146   
 created_at_parse_failed           | 0        
 closed_at_parse_failed            | 0        
 created_at_before_1900            | 0        
 closed_at_before_1900             | 101      
 future_created_at                 | 0        
 future_closed_at                  | 2        
 closed_before_created             | 47106    
 closed_status_missing_date        | 53932    
 non_closed_status_has_closed_date | 81043    
 requests_with_resolution_hours    | 21747037 
 requests_without_resolution_hours | 1014159  



In [16]:
lifecycle_quality_summary_df = (
    nyc311_lifecycle_df
    .groupBy("lifecycle_quality_status")
    .count()
    .orderBy(desc("count"))
)

lifecycle_quality_summary_df.show(truncate=False)

[Stage 4:======================================================>(118 + 2) / 120]

+---------------------------------+--------+
|lifecycle_quality_status         |count   |
+---------------------------------+--------+
|ok                               |22624409|
|closed_status_missing_closed_date|53932   |
|closed_before_created            |47106   |
|non_closed_status_has_closed_date|35646   |
|closed_at_before_1900            |101     |
|future_closed_at                 |2       |
+---------------------------------+--------+



In [17]:
status_date_summary_df = (
    nyc311_lifecycle_df
    .groupBy("status_date_consistency")
    .count()
    .orderBy(desc("count"))
)

status_date_summary_df.show(truncate=False)

[Stage 7:======================================================>(119 + 1) / 120]

+---------------------------------+--------+
|status_date_consistency          |count   |
+---------------------------------+--------+
|consistent                       |22626118|
|non_closed_status_has_closed_date|81043   |
|closed_status_missing_closed_date|53932   |
|closed_at_before_1900            |101     |
|future_closed_at                 |2       |
+---------------------------------+--------+



In [18]:
status_summary_df = (
    nyc311_lifecycle_df
    .groupBy("status")
    .count()
    .orderBy(desc("count"))
)

status_summary_df.show(truncate=False)

[Stage 10:=====================================================>(119 + 1) / 120]

+-----------+--------+
|status     |count   |
+-----------+--------+
|Closed     |22294838|
|In Progress|286243  |
|Open       |84638   |
|Pending    |63306   |
|Assigned   |24590   |
|Started    |4739    |
|Unspecified|2841    |
|Cancel     |1       |
+-----------+--------+



## Explore `nyc311_lifecycle_df`

In [19]:
nyc311_lifecycle_df.sample(
    withReplacement=False,
    fraction=0.0001,
    seed=42
).limit(5).show(truncate=False)

26/10/30 13:42:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 13:>                                                         (0 + 1) / 1]

+----------+----------------------+----------------------+------+--------------------------------------------------+---------------------------------+------------------------------------+-----------------------------+--------------------------+------------+--------------------+----------------+---------------+-------------------+---------------------+---------------------+------------+----------------+--------------+-------------+-----------+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------